# Verificação da licença do Gurobi

Este notebook verifica se o `gurobipy` está disponível, se o Gurobi consegue inicializar uma licença e resolver modelos LP e MILP. O teste opcional de tamanho tenta resolver um modelo com 2.001 variáveis e 2.001 restrições para detectar o limite comum de licenças restritas.

**Importante:** passar nos testes confirma que o solver funciona neste ambiente e que o modelo de teste maior foi aceito; isso não autentica a origem acadêmica da licença nem garante ausência de outros limites. Não são exibidos arquivos de licença, variáveis de ambiente ou mensagens completas que possam conter dados sensíveis. Execute as células em ordem.

In [1]:
try:
    import gurobipy as gp
    from gurobipy import GRB
except ImportError:
    gp = None
    GRB = None
    import_status = "pacote_ausente"
    print("FALHA: gurobipy não está instalado no kernel deste notebook.")
else:
    import_status = "disponivel"
    print(f"gurobipy disponível; versão: {gp.gurobi.version()}")

env = None
license_status = "nao_testada"
if gp is not None:
    try:
        env = gp.Env(empty=True)
        env.setParam("OutputFlag", 0)
        env.start()
    except gp.GurobiError as exc:
        license_status = "falha_de_inicializacao"
        print(f"FALHA: não foi possível inicializar a licença (código {exc.errno}).")
        if env is not None:
            env.dispose()
            env = None
    else:
        license_status = "inicializada"
        print("OK: ambiente do Gurobi inicializado com uma licença disponível.")

gurobipy disponível; versão: (13, 0, 3)
OK: ambiente do Gurobi inicializado com uma licença disponível.


In [2]:
lp_status = "nao_executado"
milp_status = "nao_executado"

if env is not None:
    try:
        with gp.Model("teste_lp", env=env) as model:
            x = model.addVar(lb=0, name="x")
            y = model.addVar(lb=0, name="y")
            model.setObjective(3 * x + 2 * y, GRB.MAXIMIZE)
            model.addConstr(x + y <= 4)
            model.addConstr(x <= 2)
            model.addConstr(y <= 3)
            model.optimize()
            lp_status = "aprovado" if model.Status == GRB.OPTIMAL and abs(model.ObjVal - 10) < 1e-6 else "falhou"
        print(f"LP: {lp_status} (ótimo esperado: 10).")
    except gp.GurobiError as exc:
        lp_status = "erro_gurobi"
        print(f"LP: falhou com erro do Gurobi (código {exc.errno}).")

    try:
        with gp.Model("teste_milp", env=env) as model:
            x = model.addVar(vtype=GRB.BINARY, name="x")
            y = model.addVar(vtype=GRB.BINARY, name="y")
            model.setObjective(x + y, GRB.MAXIMIZE)
            model.addConstr(x + y <= 1.5)
            model.optimize()
            milp_status = "aprovado" if model.Status == GRB.OPTIMAL and abs(model.ObjVal - 1) < 1e-6 else "falhou"
        print(f"MILP: {milp_status} (ótimo esperado: 1).")
    except gp.GurobiError as exc:
        milp_status = "erro_gurobi"
        print(f"MILP: falhou com erro do Gurobi (código {exc.errno}).")
else:
    print("LP e MILP não executados: não há licença inicializada.")

LP: aprovado (ótimo esperado: 10).
MILP: aprovado (ótimo esperado: 1).


In [3]:
# Este teste é pequeno em uso de memória, mas pode levar alguns segundos.
size_test_status = "nao_executado"

if env is not None:
    try:
        with gp.Model("teste_limite_tamanho", env=env) as model:
            x = model.addVars(2001, lb=0, ub=1, name="x")
            model.setObjective(gp.quicksum(x.values()), GRB.MAXIMIZE)
            model.addConstrs((x[i] <= 1 for i in range(2001)), name="bound")
            model.optimize()
            size_test_status = "aprovado" if model.Status == GRB.OPTIMAL else "falhou"
        print(f"Teste de 2.001 variáveis e restrições: {size_test_status}.")
    except gp.GurobiError as exc:
        message = str(exc).lower()
        size_limit_indicators = ("size-limited", "size limited", "too large", "limit on the number")
        if any(indicator in message for indicator in size_limit_indicators) or exc.errno == 10010:
            size_test_status = "limite_de_tamanho_detectado"
            print("Teste de tamanho: licença ou configuração impôs um limite ao modelo de teste.")
        else:
            size_test_status = "erro_gurobi"
            print(f"Teste de tamanho: falhou com erro do Gurobi (código {exc.errno}).")
else:
    print("Teste de tamanho não executado: não há licença inicializada.")

Teste de 2.001 variáveis e restrições: aprovado.


In [4]:
print("Resumo da verificação")
print(f"- Pacote gurobipy: {import_status}")
print(f"- Inicialização da licença: {license_status}")
print(f"- Otimização LP: {lp_status}")
print(f"- Otimização MILP: {milp_status}")
print(f"- Modelo com mais de 2.000 variáveis/restrições: {size_test_status}")

if env is not None:
    env.dispose()

print("\nEste diagnóstico testa funcionalidade local; confirme no portal Gurobi ou com o administrador se a licença é acadêmica e está atribuída corretamente.")

Resumo da verificação
- Pacote gurobipy: disponivel
- Inicialização da licença: inicializada
- Otimização LP: aprovado
- Otimização MILP: aprovado
- Modelo com mais de 2.000 variáveis/restrições: aprovado

Este diagnóstico testa funcionalidade local; confirme no portal Gurobi ou com o administrador se a licença é acadêmica e está atribuída corretamente.
